# 08 · Demanda: cuánto dinero se gasta en comer fuera en cada zona

**Objetivo:** estimar, por hexágono, el **gasto anual en alimentos fuera de casa** que puede capturar un restaurante. Tiene dos partes:

| Parte | Quién gasta | Dónde se asigna | Con qué datos |
|---|---|---|---|
| **Residentes** | Los hogares que viven en la zona | Hexágono donde viven | ENIGH 2024 + Censo 2020 |
| **Trabajadores** | Quienes trabajan en la zona (comida del día laboral) | Hexágono donde trabajan | ENIGH 2024 + empleo estimado del DENUE |

**Cómo:**

1. Con los microdatos de la ENIGH 2024 (hogares urbanos) se ajusta una regresión lineal del gasto anual en comer fuera contra variables que también existen en el Censo 2020: integrantes ocupados y no ocupados, escolaridad, si tienen auto y si viven en Querétaro.
2. Como el modelo es **lineal**, el gasto de un hogar se puede separar en lo que aporta cada persona ocupada (que se asigna al lugar de trabajo) y el resto (que se asigna a la vivienda).
3. Esos coeficientes se aplican a la población del Censo 2020 y al empleo del DENUE de cada hexágono.

El resultado queda en pesos de 2024 y se compara con las ventas de los Censos Económicos (pesos de 2023) solo como orden de magnitud.

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import h3
import mapas

ROOT = Path.cwd().parent
PROC, FIG = ROOT / "data/processed", ROOT / "figures"
RES = 8
EMPLEO_PUNTO_MEDIO = {"0 a 5 personas": 3, "6 a 10 personas": 8, "11 a 30 personas": 20, "31 a 50 personas": 40,
                      "51 a 100 personas": 75, "101 a 250 personas": 175, "251 y más personas": 300}

# Años de estudio aproximados de cada nivel de educa_jefe (ENIGH)
ANIOS_EDUCA = {1: 0, 2: 0, 3: 3, 4: 6, 5: 7.5, 6: 9, 7: 10.5, 8: 12, 9: 14, 10: 16, 11: 18}

## 1. Preparar la ENIGH

- `gasto_fuera` = `ali_fuera` × 4 (la ENIGH lo reporta por trimestre).
- `no_ocupados` = integrantes − ocupados.
- `escolaridad` = años aproximados del nivel educativo del jefe del hogar.
- `auto` = 1 si el hogar tiene al menos un automóvil.
- `queretaro` = 1 si el hogar está en Querétaro: ajusta el nivel de gasto a la entidad.

In [ ]:
enigh = pd.read_parquet(PROC / "enigh2024_hogares_urbanos.parquet")
enigh["gasto_fuera"] = enigh["ali_fuera"].fillna(0) * 4
enigh["no_ocupados"] = (enigh["tot_integ"] - enigh["ocupados"]).clip(lower=0)
enigh["escolaridad"] = enigh["educa_jefe"].map(ANIOS_EDUCA)
enigh["auto"] = (enigh["num_auto"].fillna(0) > 0).astype(int) if "num_auto" in enigh else 0
enigh["queretaro"] = (enigh["entidad"] == 22).astype(int)
enigh = enigh.dropna(subset=["gasto_fuera", "ocupados", "no_ocupados", "escolaridad", "factor"])

w = enigh["factor"]
print(f"Hogares: {len(enigh):,} · gasto anual promedio en comer fuera: ${np.average(enigh['gasto_fuera'], weights=w):,.0f}")
qro = enigh[enigh["queretaro"] == 1]
if len(qro):
    print(f"Querétaro: {len(qro):,} hogares · promedio: ${np.average(qro['gasto_fuera'], weights=qro['factor']):,.0f}")

## 2. Modelo de gasto del hogar

Mínimos cuadrados ponderados por el factor de expansión, con errores robustos (HC1). Cada coeficiente se lee en **pesos al año**: por ejemplo, el de `ocupados` es cuánto más gasta en comer fuera un hogar por cada integrante que trabaja.

In [ ]:
usa_auto = enigh["auto"].nunique() > 1
formula = "gasto_fuera ~ ocupados + no_ocupados + escolaridad + queretaro" + (" + auto" if usa_auto else "")
modelo = smf.wls(formula, data=enigh, weights=enigh["factor"]).fit(cov_type="HC1")
coef = modelo.params
print(f"R² = {modelo.rsquared:.3f} · hogares = {int(modelo.nobs):,}")
pd.DataFrame({"pesos al año": coef, "inf_95": modelo.conf_int()[0], "sup_95": modelo.conf_int()[1], "p": modelo.pvalues}).round(2)

## 3. Empleo por hexágono (2020 y 2025)

El empleo estimado suma el punto medio del rango de personal de todos los establecimientos, incluido el gobierno.

In [ ]:
def a_hex(df):
    return df.assign(h3=[h3.latlng_to_cell(la, lo, RES) for la, lo in zip(df["latitud"], df["longitud"])])

def empleo(edicion):
    den = pd.concat([pd.read_parquet(PROC / f"denue_zmq_{edicion}.parquet", columns=["latitud", "longitud", "per_ocu"]),
                     pd.read_parquet(PROC / f"gobierno_zmq_{edicion}.parquet", columns=["latitud", "longitud", "per_ocu"])])
    den = a_hex(den)
    return den["per_ocu"].map(EMPLEO_PUNTO_MEDIO).fillna(3).groupby(den["h3"]).sum()

emp = pd.DataFrame({"empleo_2020": empleo("2020-04"), "empleo_2025": empleo("2025-05")}).fillna(0)
emp.sum().round(0)

## 4. Demanda por hexágono

Con los coeficientes del modelo:

- **Residentes** = viviendas × (constante + Querétaro + escolaridad promedio × coef.) + viviendas con auto × coef. de auto + no ocupados × coef. de no ocupados
- **Trabajadores** = empleo del hexágono × coef. de ocupados

La población es del Censo 2020 en los dos años; lo que cambia entre 2020 y 2025 es el empleo.

In [ ]:
base = pd.read_parquet(PROC / f"hex_res{RES}_base_2020.parquet").set_index("h3")
esc = base["escolaridad"].fillna(base["escolaridad"].median())
residentes = (base["TVIVHAB"].fillna(0) * (coef["Intercept"] + coef["queretaro"] + coef["escolaridad"] * esc)
              + coef["no_ocupados"] * (base["POBTOT"].fillna(0) - base["POCUPADA"].fillna(0)).clip(lower=0))
if usa_auto:
    residentes += coef["auto"] * base["VPH_AUTOM"].fillna(0)

dem = pd.DataFrame({"demanda_residentes": residentes.clip(lower=0)}).join(emp, how="outer").fillna(0)
for anio in ("2020", "2025"):
    dem[f"demanda_trabajadores_{anio}"] = coef["ocupados"] * dem[f"empleo_{anio}"]
    dem[f"demanda_{anio}"] = dem["demanda_residentes"] + dem[f"demanda_trabajadores_{anio}"]

def entorno(tabla, cols):
    idx = set(tabla.index)
    return pd.DataFrame({c: tabla.loc[[v for v in h3.grid_disk(c, 1) if v in idx], cols].sum() for c in tabla.index}).T.add_suffix("_entorno")

dem = dem.join(entorno(dem, ["demanda_2020", "demanda_2025"]))
dem.rename_axis("h3").reset_index().to_parquet(PROC / f"demanda_hex{RES}.parquet", index=False)
tot = dem[["demanda_residentes", "demanda_trabajadores_2025", "demanda_2025"]].sum() / 1e6
print("Demanda anual estimada en la ZM (millones de pesos):")
tot.round(0)

## 5. Revisión de orden de magnitud

Se compara el gasto estimado de residentes y trabajadores con las ventas de restaurantes de los Censos Económicos y con las ventas estimadas en el notebook 07. No tienen que coincidir: los restaurantes también venden a turistas y visitantes, y el gasto en comer fuera incluye puestos y lugares que no son restaurantes. Pero deberían ser del mismo orden.

In [ ]:
oferta = pd.read_parquet(PROC / f"oferta_hex{RES}.parquet").set_index("h3")
ce = pd.read_parquet(PROC / "ce2024_restaurantes.parquet")
ventas_censo = ce[(ce["codigo"] == "722") & ce["cve_mun"].isin([6, 8, 11, 14])]["m000a"].sum()
pd.Series({"Demanda estimada 2025 (ENIGH)": dem["demanda_2025"].sum() / 1e6,
           "Ventas estimadas 2025 (notebook 07)": oferta["oferta_2025"].sum() / 1e6,
           "Ventas restaurantes, Censos Económicos (2023)": ventas_censo}, name="millones de pesos").round(0)

## 6. Mapa · Demanda anual por hexágono, 2025

In [ ]:
v = dem["demanda_2025"]
v = v[v > 0] / 1e6
BORDES = [0, 1, 5, 20, 50, 150, max(151, v.max() + 1)]
ETIQ = ["< 1", "1–5", "5–20", "20–50", "50–150", "150 o más"]
cmap, norm = mapas.escala_por_clases(BORDES, ["#EAF1F6", "#BFD6E6", "#8DB7D3", "#4F8DB8", "#2E6F95", "#14213D"])
fig, ax, polys = mapas.base_mapa(v.index)
coll = mapas.capa_hex(ax, polys, array=v.values, cmap=cmap, norm=norm)
mapas.referencias(ax)
mapas.barra_clases(fig, ax, coll, BORDES, ETIQ, "Gasto anual en comer fuera (millones de pesos)")
ax.set_title("Demanda: gasto de residentes y trabajadores · 2025", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "08_mapa_demanda.png", dpi=200, bbox_inches="tight"); plt.show()

## 7. Mapa · Saturación: oferta / demanda en el entorno

Valores **mayores a 1** indican zonas donde los restaurantes venden más de lo que gastan sus residentes y trabajadores (atraen gente de fuera o están saturadas); **menores a 1**, zonas donde el gasto local supera lo que venden los restaurantes cercanos.

In [ ]:
sat = (oferta["oferta_2025_entorno"] / dem["demanda_2025_entorno"].replace(0, np.nan)).dropna()
sat = sat[dem.loc[sat.index, "demanda_2025_entorno"] > 1e6]          # solo zonas con al menos $1 millón de demanda
BORDES = [0, 0.25, 0.5, 1, 2, 4, max(4.1, sat.max() + 0.1)]
ETIQ = ["< 0.25", "0.25–0.5", "0.5–1", "1–2", "2–4", "≥ 4"]
cmap, norm = mapas.escala_por_clases(BORDES, ["#2E6F95", "#8DB7D3", "#EAF1F6", "#FDD49E", "#F16913", "#7F2704"])
fig, ax, polys = mapas.base_mapa(sat.index)
coll = mapas.capa_hex(ax, polys, array=sat.values, cmap=cmap, norm=norm)
mapas.referencias(ax)
mapas.barra_clases(fig, ax, coll, BORDES, ETIQ, "Ventas de restaurantes / gasto local (entorno)")
ax.set_title("Saturación de la oferta · 2025", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "08_mapa_saturacion.png", dpi=200, bbox_inches="tight"); plt.show()
print(f"Mediana de saturación: {sat.median():.2f}")